readme_content = f"""
# Day 4 — Model Tuning & Reproducible Pipeline

## How to Reproduce Results
1. Run all cells in order (Run → Run All Cells)
2. All random states fixed to `42`
3. Expected precision: ~0.77-0.78, ROC AUC: ~0.92-0.93

## Library Versions Used
- scikit-learn: {sklearn.__version__}
- pandas: {pd.__version__}
- numpy: {np.__version__}
- joblib: {joblib.__version__}

## Reproducibility Checklist
- [x] All `random_state=42` in models, CV, search
- [x] numpy/random seeds fixed at top
- [x] Pipeline ensures no data leakage
- [x] Dense matrix for HGB (`sparse_output=False`)
- [x] Feature selection (k=30) from Day 3 MI scores

## Special Notes
- HGB requires dense matrix → OneHotEncoder(sparse_output=False)
- Feature engineering preserves 14 original + 7 engineered = 21 cols
- SelectKBest(k=30) keeps top features by MI from Day 3
- fnlwgt dropped (sampling weight), has_capital_gain dropped (redundant)
"""





In [3]:
# TASK 1: BUILD FULLY REPRODUCIBLE PIPELINES

# ─── SECTION 1: IMPORTS & GLOBAL SEEDS ───
import sklearn
import pandas as pd
import numpy as np
import joblib
import random
import warnings
warnings.filterwarnings('ignore')  # Hide warnings for cleaner output

# FIX ALL RANDOMNESS SOURCES - THIS IS THE KEY TO REPRODUCIBILITY
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)      # NumPy's random number generator
random.seed(RANDOM_SEED)         # Python's built-in random module

# ─── SECTION 2: LIBRARY VERSIONS (REPRODUCIBILITY RECORD) ───
# Different library versions = different results. ALWAYS record these.
print("=== LIBRARY VERSIONS (for reproducibility) ===")
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")
print(f"numpy: {np.__version__}")
print(f"joblib: {joblib.__version__}")
print(f"Random seed: {RANDOM_SEED}")

# ─── SECTION 3: LOAD DATA & SPLIT (SAME AS DAY 1-3) ───
# Load from UCI repository
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
columns = ['age','workclass','fnlwgt','education','education_num','marital_status',
           'occupation','relationship','race','sex','capital_gain','capital_loss',
           'hours_per_week','native_country','income']
df = pd.read_csv(url, names=columns, na_values=' ?', skipinitialspace=True)
df.dropna(inplace=True)  # Remove rows with missing values (marked as ' ?')

# SPLIT FEATURES (X) FROM TARGET (y)
X = df.drop('income', axis=1)              # All columns EXCEPT income
y = (df['income'] == '>50K').astype(int)   # Target: 1 if >50K, 0 if ≤50K

# SPLIT INTO TRAIN (80%) AND TEST (20%) - STRATIFIED TO KEEP CLASS BALANCE
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,                    # 20% for test
    random_state=RANDOM_SEED,         # FIXED SEED = same split every time
    stratify=y                        # KEEP 76/24 CLASS BALANCE IN BOTH SETS
)
print(f"\nTrain: {X_train.shape}, Test: {X_test.shape}")
print(f"Train positive rate: {y_train.mean():.4f}, Test: {y_test.mean():.4f}")

# ─── SECTION 4: FEATURE ENGINEERING FUNCTION (PRESERVES ORIGINAL COLUMNS) ───
# This function takes raw data (14 cols) and returns 21 columns (14 original + 7 new)
def engineer_features(df):
    """
    Create 7 engineered features from raw columns.
    IMPORTANT: Returns df.copy() + new columns = 21 total (NOT just 7 new ones!)
    """
    out = df.copy()  # KEEP ALL 14 ORIGINAL COLUMNS
    
    # 1. Binary flag: does this person have ANY capital gain?
    out['has_capital_gain'] = (df['capital_gain'] > 0).astype(int)
    
    # 2. Log-transformed capital gain (compresses huge range 0-99999)
    out['log_capital_gain'] = np.log1p(df['capital_gain'])
    
    # 3. Binary flag: any capital loss?
    out['has_capital_loss'] = (df['capital_loss'] > 0).astype(int)
    
    # 4. Age groups (life cycle: young → peak earning → retired)
    out['age_group'] = pd.cut(df['age'], bins=[0,24,39,54,64,100],
                              labels=['<25','25-39','40-54','55-64','65+'])
    
    # 5. Hours categories (part-time → standard → full-time → overtime)
    out['hours_category'] = pd.cut(df['hours_per_week'], bins=[0,19,39,49,100],
                                   labels=['<20','20-39','40-49','50+'])
    
    # 6. Higher education flag (Bachelor's or above = education_num >= 14)
    out['higher_ed'] = (df['education_num'] >= 14).astype(int)
    
    # 6. Interaction: education × hours (highly educated + long hours = high earners)
    out['edu_hours_interaction'] = df['education_num'] * df['hours_per_week']
    
    return out  # Returns 21 columns: 14 original + 7 new

# ─── SECTION 3: DEFINE COLUMN LISTS FOR PREPROCESSOR ───
# After engineer_features(): 21 columns total
# We tell ColumnTransformer which are numeric vs categorical

numeric_cols = [
    'age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss',
    'hours_per_week', 
    'has_capital_gain', 'log_capital_gain', 'has_capital_loss',
    'higher_ed', 'edu_hours_interaction'
]  # 11 numeric columns

categorical_cols = [
    'workclass', 'education', 'marital_status', 'occupation',
    'relationship', 'race', 'sex', 'native_country',
    'age_group', 'hours_category'
]  # 10 categorical columns

# ─── SECTION 4: DEFINE PREPROCESSING PIPELINES ───
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

# NUMERIC: fill missing with median, then standardize (mean=0, std=1)
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),  # Fill NaN with median
    ('scaler', StandardScaler())                    # Standardize: (x - mean) / std
])

# CATEGORICAL: fill missing with most frequent, then OneHot encode
# sparse_output=False → HGB needs DENSE matrix (not sparse)
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),  # Fill NaN with mode
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# COMBINE: ColumnTransformer applies correct pipeline to correct columns
preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

# ─── SECTION 5: FEATURE SELECTION (FROM DAY 3 TASK 5) ───
# Keep top 30 features by Mutual Information score
from sklearn.feature_selection import SelectKBest, mutual_info_classif
selector = SelectKBest(mutual_info_classif, k=30)

# ─── SECTION 6: ENGINEERING STEP (WRAPS OUR FUNCTION) ───
# FunctionTransformer wraps our engineer_features() so it fits in Pipeline
engineer = FunctionTransformer(engineer_features)

# ─── SECTION 7: BASE MODELS WITH FIXED RANDOM STATE ───
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression

RANDOM_SEED = 42

# HGB: Our Day 3 winner. random_state fixes histogram binning & sampling.
hgb_model = HistGradientBoostingClassifier(random_state=RANDOM_SEED)

# RF: 100 trees, parallel (n_jobs=-1), reproducible with random_state
rf_model = RandomForestClassifier(n_estimators=100, random_state=RANDOM_SEED, n_jobs=-1)

# LR: liblinear solver supports L1/L2, reproducible with random_state
lr_model = LogisticRegression(solver='liblinear', random_state=RANDOM_SEED, max_iter=1000)

# ─── SECTION 8: COMPLETE PIPELINES FOR ALL 3 MODELS ───
# Each pipeline: Engineer → Preprocess → Select(k=30) → Model
# Only the MODEL step differs — everything else IDENTICAL for fair comparison

pipelines = {
    'HistGradientBoosting': Pipeline([
        ('engineer', FunctionTransformer(engineer_features)),
        ('preprocessor', preprocessor),
        ('select', SelectKBest(mutual_info_classif, k=30)),
        ('model', hgb_model)
    ]),
    'RandomForest': Pipeline([
        ('engineer', FunctionTransformer(engineer_features)),
        ('preprocessor', preprocessor),
        ('select', SelectKBest(mutual_info_classif, k=30)),
        ('model', rf_model)
    ]),
    'LogisticRegression': Pipeline([
        ('engineer', FunctionTransformer(engineer_features)),
        ('preprocessor', preprocessor),
        ('select', SelectKBest(mutual_info_classif, k=30)),
        ('model', lr_model)
    ])
}

# ─── SECTION 9: SANITY CHECK — VERIFY PIPELINE STRUCTURE ───
# Fit one pipeline to see feature counts
print("\n=== PIPELINE STRUCTURE VERIFICATION ===")
test_pipe = pipelines['HistGradientBoosting']
test_pipe.fit(X_train, y_train)
n_features = len(test_pipe.named_steps['preprocessor'].get_feature_names_out())
print(f"Original columns: 14")
print(f"After engineer_features(): 21 (14 original + 7 engineered)")
print(f"After preprocessor (OneHot): ~122 features")
print(f"After SelectKBest(k=30): {test_pipe.named_steps['select'].get_support().sum()} features")
print("✅ Pipeline structure verified!")



=== LIBRARY VERSIONS (for reproducibility) ===
scikit-learn: 1.6.1
pandas: 2.3.3
numpy: 2.1.1
joblib: 1.5.2
Random seed: 42

Train: (26048, 14), Test: (6513, 14)
Train positive rate: 0.2408, Test: 0.2407

=== PIPELINE STRUCTURE VERIFICATION ===
Original columns: 14
After engineer_features(): 21 (14 original + 7 engineered)
After preprocessor (OneHot): ~122 features
After SelectKBest(k=30): 30 features
✅ Pipeline structure verified!
